# Qwen3.5-35B-A3B Uncensored - Kaggle GPU

Kaggle T4 x2 (30GB VRAM) で Qwen3.5-35B-A3B を動かすノートブック。

**設定:** Settings > Accelerator > GPU T4 x2 を選択してから実行してください。

In [ ]:
# 1. 依存関係インストール
!pip install -q llama-cpp-python huggingface-hub gradio

In [ ]:
# 2. GPU確認
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB")

In [ ]:
# 3. モデルダウンロード (Q4_K_M ~20GB)
from huggingface_hub import hf_hub_download

REPO_ID = "HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive"
FILENAME = "Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"Model downloaded: {model_path}")

In [ ]:
# 4. モデル読み込み
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_gpu_layers=-1,  # 全レイヤーGPUオフロード
    verbose=False,
)
print("Model loaded!")

In [ ]:
# 5. チャットUI起動
import gradio as gr

def respond(message, history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repeat_penalty=1.5,
    )
    return response["choices"][0]["message"]["content"]

demo = gr.ChatInterface(
    fn=respond,
    title="Qwen3.5-35B-A3B Chat",
    description="スマホからそのまま使えるチャットUI",
    examples=["こんにちは！自己紹介してください。", "日本語で短い物語を書いて", "Pythonでフィボナッチ数列を書いて"],
    retry_btn="リトライ",
    undo_btn="取り消し",
    clear_btn="リセット",
)
demo.launch(share=True)  # share=True で公開URL発行